In [ ]:
# Imports

## Core libraries
import numpy as np
import matplotlib.pyplot as plt

## TensorFlow / Keras
import tensorflow as tf
import keras
from keras import ops, layers
from keras.callbacks import Callback
from keras.models import Sequential, Model
from keras.layers import Input, Conv2D, Dense


In [ ]:
# Setting seeds

np.random.seed(1999)
tf.random.set_seed(1999)


In [ ]:
# Loading the dataset

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

## Feature scaling (mapping pixel values to [0,1])
'''
This is used as neural network training via gradient descent is sensitive to the
scale of inputs (through activations a = Wx + b). This causes gradients to vanish
and makes the loss landscape poorly conditioned.
'''
x_train_full = x_train_full / 255.0
x_test = x_test / 255.0

''' Recalling our y values are our categories '''
y_train_full = keras.utils.to_categorical(y_train_full, 10)
y_test = keras.utils.to_categorical(y_test, 10)

## Setting up our validation dataset
n_val = int(0.2 * len(x_train_full))
perm = np.random.permutation(len(x_train_full))
val_idx, train_idx = perm[:n_val], perm[n_val:]
x_val, y_val = x_train_full[val_idx], y_train_full[val_idx]
x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]

## Check
print(f"train: {x_train.shape} val: {x_val.shape} test: {x_test.shape}")


In [ ]:
# Augmenting the dataset
'''
This involves creating new training examples by applying label preserving
transformations to existing examples e.g. flips, crops, scaling, colour jitter,
adding noise, ...
'''

'''
Applying three transformations in sequence, we pad the image with black 0s,
then randomly crop the newly padded image (so the model learns that a shifted
horse picture is still a horse), and then randomly mirroring the image from left
to right.
'''
augment = keras.Sequential([layers.ZeroPadding2D(padding=4),
                            layers.RandomCrop(32, 32),
                            layers.RandomFlip("horizontal")])

batch_size = 128

'''
This builds the full training data pipeline, where:
- The from_tensor_slices((x_train, y_train)) wraps the numpy arrays into a
  tf.data.Dataset that has image label pairs one at a time.
- For .shuffle(len(x_train)) - this shifts the entire dataset after each epoch.
- For .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
  this applies the augmentation pipeline to each batch of images whilst leaving
  labels untouched. The training=True is important because RandomCrop and
  RandomFlip only apply their transformations during training. The
  num_parallel_calls=tf.data.AUTOTUNE lets TensorFlow decide how many batches to
  augment in parallel across CPU threads.
'''
train_ds = (tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(len(x_train)).batch(batch_size).map(lambda x, y: (augment(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE))

### Recalling that there is no augmentation for validation and test datasets
val_ds  = (tf.data.Dataset.from_tensor_slices((x_val,  y_val ))
           .batch(batch_size).prefetch(tf.data.AUTOTUNE))
test_ds = (tf.data.Dataset.from_tensor_slices((x_test, y_test))
           .batch(batch_size).prefetch(tf.data.AUTOTUNE))


In [ ]:
# Residual block (no dropout)
'''
Here is the specification requirements for a single residual block (the repeating
unit in ResNet-20). Instead of learning the mapping between the input and the
output, we instead learn the difference between the input and the desired output
(the residual).

In this, the shortcut path passes the input unchanged (albeit with a channel
count change to match shapes at times). These two paths are added together for
a final ReLU.
'''

### We recall that a convolution slides a small window (a kernel/filter) across our grid.
### A single kernel produces one output grid (one channel). One kernel might learn to detect vertical edges, another colour gradients, and so on.
### We thus use 16 kernels simultaneously to produce an output with 16 channels.

class ResidualBlock(keras.layers.Layer):

    '''
    The init method constructs everything which doesn't fit into the other categories.
    In this, we construct the two convolutional layers and the ReLU activation.
    Everything here is created once and reused when the block processes an input.
    Projection is handled separately, as the input's channel count changes.
    '''
    def __init__(self, filters, stride=1, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.stride = stride

        ### The same padding here adds just enough padding at each convolution step so the output spatial dimensions match the input.
        ### The kernel initialiser establishes starting weights for the kernels such that they are not all zero (each kernel learns the same thing), too large (where activations explode), or too small (vanishing to zero).
        self.conv1 = layers.Conv2D(filters, 3, strides=stride, padding="same",
                                   kernel_regularizer=keras.regularizers.L2(1e-4),
                                   kernel_initializer="he_normal")

        ### Batch normalisation ensures each channel has a mean of 0 and a variance of 1.
        ### This is necessary for faster and more stable training as the weights update during training, causing our outputs to shift considerably (and calibrated to the old weights).
        ### The purpose of batch normalisation is to ensure a stable range of outputs.
        self.bn1 = layers.BatchNormalization()
        self.conv2 = layers.Conv2D(filters, 3, padding="same",
                                   kernel_regularizer=keras.regularizers.L2(1e-4),
                                   kernel_initializer="he_normal")
        self.bn2 = layers.BatchNormalization()
        self.relu = layers.Activation("relu")

    ### Going deeper into the network, we want to detect increasingly abstract features.
    ### We therefore downsample (have a bigger stride) as we detect combinations.
    ### We compensate for that by increasing the number of channels.

    '''
    The build method handles the projection shortcut (and handles shape-matching if needed).
    '''
    def build(self, input_shape):
        in_channels = input_shape[-1]

        if self.stride != 1 or in_channels != self.filters:
            self.proj = layers.Conv2D(self.filters, 1, strides=self.stride, padding="valid",
                                      kernel_regularizer=keras.regularizers.L2(1e-4),
                                      kernel_initializer="he_normal")
            self.bn_proj = layers.BatchNormalization()
        else:
            self.proj = None

        super().build(input_shape)

    '''
    The call method runs the actual forward pass of the model organising the
    split into the main convolutional path and the shortcut path, then summing
    them and applying ReLU.
    '''
    def call(self, x):
        if self.proj is not None:
            shortcut = self.bn_proj(self.proj(x))
        else:
            shortcut = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        return self.relu(out + shortcut)

    '''
    The get_config method enables serialisation so Keras can save/load the model.
    '''
    def get_config(self):
        config = super().get_config()
        config.update({"filters": self.filters,
                       "stride": self.stride})
        return config


In [ ]:
# build_resnet20 — deep ensemble variant (same deterministic member network)
'''
The full architecture follows the ResNet-20 spec of He et al. (2016) for CIFAR:
an initial conv, three stages of three residual blocks at filter widths
{16, 32, 64}, with the first block of stages 2 and 3 using stride 2 to halve
spatial resolution while doubling channels.

For the deep ensemble baseline, each member is this same deterministic ResNet-20.
The uncertainty signal comes from training five networks independently, then
averaging their predictive probabilities at inference time.
'''
def build_resnet20():
    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Conv2D(16, 3, padding="same",
                      kernel_regularizer=keras.regularizers.L2(1e-4),
                      kernel_initializer="he_normal")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    for _ in range(3):
        x = ResidualBlock(16)(x)

    x = ResidualBlock(32, stride=2)(x)
    for _ in range(2):
        x = ResidualBlock(32)(x)

    x = ResidualBlock(64, stride=2)(x)
    for _ in range(2):
        x = ResidualBlock(64)(x)

    ### Global average pooling takes the block's output and feeds it into a final dense layer for classification.
    ### It takes the 64 feature maps (each of which is an 8x8 grid) and averages each grid into a single number. The output is a flat vector of length 64, that feeds into the final dense layer for classification.
    ### This is therefore translation invariant (if a feature shifts position in the 8x8 grid, the average barely changes), so classification is stable.
    x = layers.GlobalAveragePooling2D()(x)

    outputs = layers.Dense(10, kernel_initializer="he_normal")(x)

    return keras.Model(inputs, outputs)


In [ ]:
# Deep ensemble settings

'''
Deep ensembles train multiple deterministic networks independently. Here we use
five ResNet-20 members, following the baseline described by Lakshminarayanan
et al. (2017). The members have the same architecture and optimiser, but use
different random seeds and separate checkpoints.
'''
ensemble_size = 5
base_seed = 1999

ensemble_models = []
ensemble_histories = []
ensemble_test_metrics = []


In [ ]:
# Training schedule for each ensemble member

'''
Step-wise learning-rate schedule: drop by 10x at epochs 100 and 150.
This is the standard CIFAR ResNet schedule from He et al. (2016) and matches
the deterministic baseline, so the baseline differs only in the ensemble
mechanism, not in the optimisation recipe.
'''
def lr_schedule(epoch, lr):
    if epoch in (100, 150):
        return lr * 0.1
    return lr


In [ ]:
# Compiling helper

### This doesn't run any training or touch any data, but it configures the optimiser (how weights are updated), the loss function, and optionally any metrics we want to track.
def compile_member(model):
    model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1, momentum=0.9),
                  loss=keras.losses.CategoricalCrossentropy(from_logits=True),
                  metrics=["categorical_accuracy"])
    return model


In [ ]:
# Callbacks for member-specific checkpoints

def make_callbacks(member_id):
    checkpoint_path = f"resnet20_ensemble_member_{member_id + 1}_best.weights.h5"
    return [keras.callbacks.ModelCheckpoint(checkpoint_path,
                                            monitor="val_loss",
                                            save_best_only=True,
                                            save_weights_only=True),
            keras.callbacks.LearningRateScheduler(lr_schedule, verbose=1)] # calls lr_schedule function each epoch to reduce learning rate


In [ ]:
# Training the 5-member deep ensemble
'''
Two-phase approach, following the safe idiom for clear_session() with ensembles:

Phase 1 — training loop:
  clear_session() is called at the top of each iteration, which is safe because
  no live Keras model objects are carried across in ensemble_models at that point.
  Only lightweight Python objects (history dicts, scalar metrics) are accumulated
  across iterations. Each member's best weights are saved to disk by ModelCheckpoint.

Phase 2 — checkpoint reload:
  After the loop, a fresh clear_session() wipes all remaining graph state, then
  all five members are rebuilt and loaded from their checkpoint files into
  ensemble_models. This avoids the version-fragile implicit dependency on TF
  reference-counting that arises when live models are accumulated across
  clear_session() calls inside the loop.
'''

# ── Phase 1: train each member independently ──────────────────────────────────
for member_id in range(ensemble_size):
    print("\n" + "=" * 70)
    print(f"Training ensemble member {member_id + 1}/{ensemble_size}")
    print("=" * 70)

    # safe: no live model objects exist in ensemble_models at this point
    keras.backend.clear_session()
    np.random.seed(base_seed + member_id)
    tf.random.set_seed(base_seed + member_id)

    model = build_resnet20()
    model = compile_member(model)

    callbacks = make_callbacks(member_id)
    history = model.fit(train_ds,
                        epochs=200,
                        validation_data=val_ds,
                        callbacks=callbacks)

    checkpoint_path = f"resnet20_ensemble_member_{member_id + 1}_best.weights.h5"
    model.load_weights(checkpoint_path)

    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    print(f"Member {member_id + 1} — test loss: {test_loss:.4f}  test accuracy: {test_acc:.4f}")

    # accumulate only lightweight Python objects — no live Keras model references
    ensemble_histories.append(history)
    ensemble_test_metrics.append((test_loss, test_acc))

member_accs = np.array([acc for _, acc in ensemble_test_metrics])
print(f"\nMember test accuracies: {member_accs}")
print(f"Mean member accuracy:   {member_accs.mean():.4f} ± {member_accs.std():.4f}")

# ── Phase 2: reload all members from checkpoints into a clean session ──────────
print("\nReloading all members from checkpoints...")
keras.backend.clear_session()  # clean slate — no residual graph state
for member_id in range(ensemble_size):
    m = build_resnet20()
    compile_member(m)
    m.load_weights(f"resnet20_ensemble_member_{member_id + 1}_best.weights.h5")
    ensemble_models.append(m)
    print(f"  Loaded member {member_id + 1}")


In [ ]:
# Plotting training curves

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for member_id, history in enumerate(ensemble_histories):
    ## Plot 1
    ax1.plot(history.history["loss"], label=f"member {member_id + 1} train")
    ax1.plot(history.history["val_loss"], linestyle="--", label=f"member {member_id + 1} val")

    ## Plot 2
    ax2.plot(history.history["categorical_accuracy"], label=f"member {member_id + 1} train")
    ax2.plot(history.history["val_categorical_accuracy"], linestyle="--", label=f"member {member_id + 1} val")

ax1.set_xlabel("epoch"); ax1.set_ylabel("loss")
ax1.set_title("Deep Ensemble ResNet-20 — Loss")
ax1.legend(fontsize=8)

ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy")
ax2.set_title("Deep Ensemble ResNet-20 — Accuracy")
ax2.legend(fontsize=8)

plt.tight_layout(); plt.show()


In [ ]:
# Loading SVHN as the out-of-distribution reference set
'''
Per the proposal §3.5, SVHN (street view of house numbers) is used as the
OOD test set. SVHN images are 32x32 colour photos like CIFAR-10, but of
digits, so a CIFAR-10 trained model should ideally report higher predictive
entropy on SVHN inputs than on its in-distribution CIFAR-10 test images.

The whole SVHN test split is ~26k examples; we load it via
tensorflow_datasets and rescale to [0, 1] to match CIFAR-10 preprocessing.
'''
import tensorflow_datasets as tfds

svhn = tfds.load("svhn_cropped", split="test", as_supervised=True)
svhn_images = []
for img, _ in svhn:
    svhn_images.append(img.numpy())
x_svhn = np.stack(svhn_images).astype("float32") / 255.0

print(f"SVHN OOD set: {x_svhn.shape}")


In [ ]:
# Loading CIFAR-10-C as the second out-of-distribution reference set
'''
Per the proposal §3.5, CIFAR-10-C (Hendrycks & Dietterich 2019) is the
second OOD dataset. Unlike SVHN which represents a full domain shift,
CIFAR-10-C tests calibration under in-distribution corruption: blur, noise,
weather effects, digital artefacts, etc. This is a distinct failure mode
from domain shift and is evaluated separately.

We load all 15 standard corruption types at severity 3 (mid-level) and
concatenate them into a single flat evaluation array. Labels are the
original CIFAR-10 test labels (repeated once per corruption type), so we
can also report accuracy under corruption alongside OOD confidence.

TFDS name format: "cifar10_corrupted/{corruption_type}_{severity}"
'''

# 15 standard corruptions from Hendrycks & Dietterich (2019)
CORRUPTIONS = [
    "brightness", "contrast", "defocus_blur", "elastic_transform",
    "fog", "frost", "glass_blur", "gaussian_noise",
    "impulse_noise", "jpeg_compression", "motion_blur",
    "pixelate", "shot_noise", "snow", "zoom_blur"
]
SEVERITY = 3  # severity levels 1–5; 3 is the standard mid-level benchmark

cifar10c_images = []
cifar10c_labels = []
loaded_corruptions = []

for corruption in CORRUPTIONS:
    ds_name = f"cifar10_corrupted/{corruption}_{SEVERITY}"
    try:
        ds = tfds.load(ds_name, split="test", as_supervised=True)
        for img, label in ds:
            cifar10c_images.append(img.numpy())
            cifar10c_labels.append(label.numpy())
        loaded_corruptions.append(corruption)
    except Exception as e:
        print(f"  Warning — skipping {corruption}: {e}")

x_cifar10c = np.stack(cifar10c_images).astype("float32") / 255.0
y_cifar10c = np.array(cifar10c_labels)   # integer labels 0–9

print(f"CIFAR-10-C OOD set: {x_cifar10c.shape}")
print(f"  Severity: {SEVERITY}  |  Corruptions loaded: {len(loaded_corruptions)}/{len(CORRUPTIONS)}")

In [ ]:
# Generating predictions for the evaluation file
'''
This is the only output the downstream evaluation notebook needs from this
file. We produce softmax probabilities on:
  - the CIFAR-10 test set (in-distribution),
  - the SVHN test set (out-of-distribution, full domain shift),
  - the CIFAR-10-C test set (out-of-distribution, in-distribution corruption).

For the deep ensemble, each independently trained member contributes one
probability vector per input. The final ensemble probability is the mean
across all five members.
'''

all_probs_test     = []
all_probs_svhn     = []
all_probs_cifar10c = []

for member_id, model in enumerate(ensemble_models):
    print(f"Predicting with ensemble member {member_id + 1}/{ensemble_size}")

    ## In-distribution: CIFAR-10 test set
    logits_test = model.predict(test_ds)                          # (10000, 10)
    all_probs_test.append(tf.nn.softmax(logits_test, axis=-1).numpy())

    ## Out-of-distribution: SVHN (full domain shift)
    logits_svhn = model.predict(x_svhn, batch_size=batch_size)
    all_probs_svhn.append(tf.nn.softmax(logits_svhn, axis=-1).numpy())

    ## Out-of-distribution: CIFAR-10-C (in-distribution corruption)
    logits_cifar10c = model.predict(x_cifar10c, batch_size=batch_size)
    all_probs_cifar10c.append(tf.nn.softmax(logits_cifar10c, axis=-1).numpy())

all_probs_test     = np.stack(all_probs_test,     axis=0)  # (5, 10000, 10)
all_probs_svhn     = np.stack(all_probs_svhn,     axis=0)  # (5, N_svhn, 10)
all_probs_cifar10c = np.stack(all_probs_cifar10c, axis=0)  # (5, N_c10c, 10)

probs_test     = all_probs_test.mean(axis=0)
probs_svhn     = all_probs_svhn.mean(axis=0)
probs_cifar10c = all_probs_cifar10c.mean(axis=0)

preds_test     = np.argmax(probs_test, axis=1)
labels_test    = np.argmax(y_test, axis=1)
preds_cifar10c = np.argmax(probs_cifar10c, axis=1)

print(f"\nEnsemble test accuracy (CIFAR-10): {(preds_test == labels_test).mean():.4f}")
print(f"Accuracy under corruption (severity {SEVERITY}): {(preds_cifar10c == y_cifar10c).mean():.4f}")
print(f"all_probs_test:     {all_probs_test.shape}")
print(f"all_probs_svhn:     {all_probs_svhn.shape}")
print(f"all_probs_cifar10c: {all_probs_cifar10c.shape}")

In [ ]:
# Saving raw outputs for the evaluation file
'''
Save format (shared across all benchmarks). The evaluation notebook will
load these by filename and produce ECE, reliability diagrams, OOD AUROC,
risk-coverage and (for the BNN benchmarks) the epistemic / aleatoric
decomposition.

Keys:
  model_name         - human-readable string identifier
  probs_test         - (N_test, 10)      mean predictive dist. on CIFAR-10 test
  labels_test        - (N_test,)         true labels (integer 0–9)
  preds_test         - (N_test,)         argmax of probs_test
  probs_svhn         - (N_svhn, 10)      mean predictive dist. on SVHN OOD
  probs_cifar10c     - (N_c10c, 10)      mean predictive dist. on CIFAR-10-C OOD
  labels_cifar10c    - (N_c10c,)         true labels for CIFAR-10-C (integer 0–9)
  preds_cifar10c     - (N_c10c,)         argmax of probs_cifar10c
  cifar10c_severity  - scalar int         corruption severity used (3)
  all_probs_test     - (S, N_test, 10)   per-member probs, S=5
  all_probs_svhn     - (S, N_svhn, 10)   per-member probs, S=5
  all_probs_cifar10c - (S, N_c10c, 10)   per-member probs, S=5
  member_test_accs   - (S,)              per-member CIFAR-10 test accuracies
'''
np.savez("deep_ensemble_5_results.npz",
         model_name="deep_ensemble_5",
         # In-distribution
         probs_test=probs_test,
         labels_test=labels_test,
         preds_test=preds_test,
         # OOD — full domain shift
         probs_svhn=probs_svhn,
         all_probs_svhn=all_probs_svhn,
         # OOD — in-distribution corruption
         probs_cifar10c=probs_cifar10c,
         labels_cifar10c=y_cifar10c,
         preds_cifar10c=preds_cifar10c,
         cifar10c_severity=SEVERITY,
         # Per-member arrays for uniform eval API
         all_probs_test=all_probs_test,
         all_probs_cifar10c=all_probs_cifar10c,
         member_test_accs=member_accs)

print("Saved deep_ensemble_5_results.npz")